# CAMS Regionale Luftqualitätsvorhersage – Analyse von Schadstoffen über Deutschland am Beispiel von Ozon

## Hintergrund

Der **Copernicus Atmosphere Monitoring Service (CAMS)** stellt Daten und Vorhersagen zur Luftqualität bereit.

Diese basieren auf Atmosphärenmodellen und Beobachtungsdaten und können zur Analyse der Luftqualität in Europa genutzt werden.

In diesem Kurs wird gezeigt, wie CAMS-Vorhersagedaten heruntergeladen, verarbeitet und räumlich dargestellt werden können.

Das **CAMS National Collaboration Programme (NCP)** unterstützt die nationale Nutzung von CAMS-Produkten.
Es fördert den Austausch zwischen Nutzenden und europäischen Diensten und stärkt damit die Anwendung von Copernicus-Atmosphärendaten auf nationaler Ebene.

Auf europäischer Ebene werden zusätzliche Trainingsmaterialien zur Nutzung von CAMS-Daten bereitgestellt.  
Ein Beispiel ist der ECMWF-Trainingskurs zur regionalen Luftqualitätsvorhersage ([ECMWF CAMS Training Notebook](https://github.com/ecmwf-training/cams-training/blob/main/cams-regional-forecast.ipynb)).

## Ziel der Übung

Ziel dieser Übung ist es, CAMS-Luftqualitätsvorhersagen für Deutschland herunterzuladen, zu verarbeiten und räumlich sowie zeitlich zu analysieren.

Am Beispiel des Luftschadstoffs **Ozon (O₃)** wird gezeigt, wie aus einer **96-Stunden-Vorhersage**:

- tägliche Mittelwerte  
- tägliche Maximalwerte  

berechnet und anschließend als Karten dargestellt werden.

Zusätzlich wird eine Zeitreihe der Ozonkonzentration für einen Beispielstandort in Deutschland (Berlin) erstellt.

## Daten und mögliche Erweiterungen

Der verwendete Datensatz enthält modellbasierte Vorhersagen verschiedener Luftschadstoffe für Europa.  

Diese Daten können auch manuell über den Atmosphere Data Store heruntergeladen werden:  
https://ads.atmosphere.copernicus.eu/datasets/cams-europe-air-quality-forecasts?tab=download

Die in diesem Kurs genutzte Methodik kann auch auf andere Luftschadstoffe angewendet werden, die im europäischen Luftqualitätsindex genutzt werden.  
Dazu muss lediglich die Modellvariable im Anfrage-Parameter angepasst werden:

- **NO₂ (Stickstoffdioxid)**  
  `variables = ['nitrogen_dioxide']`

- **PM10 (Feinstaub < 10 µm)**  
  `variables = ['particulate_matter_10um']`

- **PM2.5 (Feinstaub < 2.5 µm)**  
  `variables = ['particulate_matter_2.5um']`

- **SO₂ (Schwefeldioxid)**  
  `variables = ['sulphur_dioxide']`

## Vorbereitung der Arbeitsumgebung

In [ ]:
# Cdsapi installieren
!pip install -q cdsapi

# Cartopy für Kartenplots installieren
!pip install cartopy

# NetCDF-Unterstützung installieren
!pip install netCDF4

## Download der Daten mit API-Schlüssel

Um Daten automatisch über die Atmosphere Data Store (ADS) API herunterladen zu können, wird ein persönlicher API-Schlüssel benötigt.

Dazu muss zunächst ein Benutzerkonto beim [ADS](https://ads.atmosphere.copernicus.eu/) erstellt bzw. ein bestehendes Konto verwendet werden.

Nach dem Login wird ein individueller API-Schlüssel bereitgestellt. Eine Anleitung zur Einrichtung der API finden Sie [hier](https://ads.atmosphere.copernicus.eu/how-to-api).

Der persönliche API-Schlüssel kann anschließend im folgenden Codeblock eingetragen werden, indem `######################` durch den eigenen Schlüssel ersetzt wird.

In [ ]:
# API Zugangsdaten setzen
import os

os.environ['CDSAPI_URL'] = 'https://ads.atmosphere.copernicus.eu/api'
os.environ['CDSAPI_KEY'] = '######################'

## Python-Bibliotheken importieren

In [ ]:
# CDS API für den Zugriff auf CAMS-Daten
import cdsapi

# Bibliotheken für Datenanalyse
import numpy as np
import pandas as pd
import xarray as xr

# Visualisierung und Karten
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Datums- und Zeitfunktionen
import datetime

# Warnungen beim API-Download unterdrücken
import urllib3
urllib3.disable_warnings()

## Parameter der Datenanfrage

In [ ]:
# Verzeichnis für heruntergeladene Daten
DATADIR = '.'

# CAMS Datensatz
cams_dataset = 'cams-europe-air-quality-forecasts'

# Datum des Modelllaufs
# Hier 1. Tag des 4-tägigen Zeitraums wählen
start_date = '2026-03-07'
end_date = '2026-03-07'

# Startzeit des Modelllaufs
time = '00:00'

## Vorhersageparameter

In [ ]:
# Vorhersagezeitraum (0–96 Stunden)
lead_time_start = 0
lead_time_stop = 96
step_hours = 1

leadtime_hours = list(range(
    lead_time_start,
    lead_time_stop + lead_time_start,
    step_hours
))

# Modellvariable (hier: Ozon)
variables = ['ozone']

# Modelltyp
models = ['ensemble']

# Modelllevel (0 = bodennahe Konzentration)
levels = [0]

## Datenanfrage an CAMS

In [ ]:
# Anfrage-Dictionary erstellen
request = {
    'variable': variables,
    'date': f'{start_date}/{end_date}',
    'time': f'{time}',
    'leadtime_hour': leadtime_hours,
    'type': 'forecast',
    'model': models,
    'level': levels,
    'data_format': 'netcdf',

    # geografischer Ausschnitt (Deutschland)
    # Format: [Nord, West, Süd, Ost]
    'area': [55, 5, 47, 16]
}

## Daten herunterladen

In [ ]:
# Zielpfad für den Download
target = f'{DATADIR}/o3-regional-forecast-deutschland.nc'

# Verbindung zur ADS API herstellen
c = cdsapi.Client()

# Datensatz herunterladen
c.retrieve(cams_dataset, request).download(target)

# Datensatz laden und vorbereiten

In [ ]:
# Datensatz öffnen
ds = xr.open_dataset(target)

# Datensatzstruktur anzeigen
ds

# Anzahl der Vorhersagezeitpunkte bestimmen
num_forecasts = len(leadtime_hours)

# Startdatum in Datetime umwandeln
start_day = pd.to_datetime(start_date)

# Zeitstempel für jede Vorhersagestunde erzeugen
forecast_index = start_day + pd.to_timedelta(np.arange(num_forecasts), 'h')

# Zeitdimension dem Datensatz zuweisen
ds_dt = ds.assign_coords(time=forecast_index)

# Ozonvariable auswählen
da = ds_dt['o3_conc']

# DataArray anzeigen
da

## Ozonvariable auswählen und Tageswerte berechnen

In [ ]:
# Ozonvariable auswählen
# In CAMS ist die bodennahe Ozonkonzentration als 'o3_conc' gespeichert
da = ds_dt['o3_conc']

# Tägliche Mittel- und Maximalwerte berechnen
o3_daily_max = da.resample(time='D').max()
o3_daily_mean = da.resample(time='D').mean()

## Karten der Ozonvorhersage erstellen

Im folgenden Schritt wird eine Funktion definiert, mit der räumliche Karten der
Ozonkonzentration dargestellt werden können.

Für jeden Vorhersagetag wird eine Karte erzeugt. Die Karten zeigen:

- die räumliche Verteilung der Ozonkonzentration
- eine Farbskala der Konzentrationswerte
- Küstenlinien und Ländergrenzen
- einen auf Deutschland begrenzten Kartenausschnitt

In [ ]:
# Funktion zum Darstellen der Vorhersagekarten
def plot_forecasts(xarray_da, num_forecasts, plot_title):

    # Kartenlayout erstellen (4 Karten untereinander)
    fig, axs = plt.subplots(
        4, 1,
        figsize=(10, 40),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    # Liste der verfügbaren Vorhersagezeitpunkte
    day_forecasts = xarray_da.time.values

    # Schleife über die einzelnen Vorhersagetage
    for i in range(num_forecasts):

        timestamp = day_forecasts[i]
        day = str(timestamp)[:10]

        # Daten für diesen Tag auswählen
        data = xarray_da.sel(time=day, level=0.0)

        # Kontrastanpassung der Farbskala
        vmin = float(data.quantile(0.05))
        vmax = float(data.quantile(0.95))

        # Rasterkarte der Ozonkonzentration erzeugen
        cs = axs[i].pcolormesh(
            xarray_da.longitude,
            xarray_da.latitude,
            data,
            cmap='YlGnBu',
            vmin=vmin,
            vmax=vmax,
            transform=ccrs.PlateCarree()
        )

        # Farbskala hinzufügen
        cbar = plt.colorbar(cs, ax=axs[i], fraction=0.046, pad=0.05)
        cbar.set_label('µg m⁻³')

        # Titel der Karte
        axs[i].set_title(plot_title + day)

        # Deutschland-Ausschnitt festlegen
        axs[i].set_extent([5, 16, 47, 55], crs=ccrs.PlateCarree())

        # Küstenlinien darstellen
        axs[i].coastlines(color='grey', alpha=0.7, resolution='50m',
                          linewidth=1.5)

        # Ländergrenzen hinzufügen
        axs[i].add_feature(cartopy.feature.BORDERS, edgecolor='grey',
                          linewidth=1.5)

        # Breiten- und Längengradlinien anzeigen
        axs[i].gridlines(draw_labels=False, linestyle='--')

## Karten der täglichen Mittelwerte

In [ ]:
plot_forecasts(o3_daily_mean, len(o3_daily_mean.time), 'O3 tägliche Mittelwerte ')

##Karten der täglichen Maximalwerte

In [ ]:
plot_forecasts(o3_daily_max, len(o3_daily_max.time), 'O3 tägliche Maximalwerte ')

## Zeitreihe der Ozonkonzentration für einen Beispielstandort (Berlin)

Neben der räumlichen Darstellung der Ozonkonzentration kann
auch eine Zeitreihe für einen einzelnen Standort erstellt werden.  
Dadurch lässt sich analysieren, wie sich die vorhergesagte Konzentration
über die 96-Stunden-Vorhersageperiode entwickelt.

Zunächst werden die Längengrade der CAMS-Daten von einem 0–360° Koordinatensystem in das in der
Kartographie übliche -180° bis 180° System umgerechnet.
  
Anschließend wird hier Berlin als Beispielstandort gewählt.

In [ ]:
# Längengrade von [0,360] auf [-180,180] umrechnen
# ECMWF-Daten verwenden standardmäßig ein 0–360° Längengrad-System
ds_180 = ds_dt.assign_coords(
    longitude=(((ds_dt.longitude + 180) % 360) - 180)
).sortby('longitude')

# Ozonvariable auswählen
da = ds_180['o3_conc']


# Beispielstandort definieren (Berlin)
# Koordinaten hier anpassen für andere Punkte
site_lat = 52.52
site_lon = 13.40


# Nächstgelegenen Modellgitterpunkt zum Standort auswählen
site_da = da.sel(
    latitude=site_lat,
    longitude=site_lon,
    method='nearest',
    level=0.0
)


# Zeitreihe der Ozonkonzentration darstellen
fig = plt.figure(figsize=(10,5))

site_da.plot(marker='o')

plt.title("Vorhersage der bodennahen Ozonkonzentration (O₃) – Berlin")
plt.xlabel("Zeit")
plt.ylabel("Ozonkonzentration (µg m⁻³)")

plt.grid(True)
plt.tight_layout()